In [6]:
import collections
import os
import pickle

import numpy as np
import pandas as pd
from sklearn.utils.class_weight import compute_class_weight

In [2]:
INPUT_PATH = '/data/hqvo3/Datasets/histo_cls/7_Regression_Grading_oesophageal_adenocarcinomas'

In [3]:
subset_names = os.listdir(INPUT_PATH)
subset_names.remove('LICENSE.txt')
print(subset_names)

['VALSET1_UKK', 'VALSET2_WNS', 'VALSET3_TCGA', 'VALSET4_CHA_FULL']


In [4]:
TRAIN_SET = subset_names[:-1]
TEST_SET = subset_names[-1]
print(TRAIN_SET)
print(TEST_SET)

['VALSET1_UKK', 'VALSET2_WNS', 'VALSET3_TCGA']
VALSET4_CHA_FULL


In [12]:
dict_of_classname_to_filenames = {}

In [5]:
test_set_path = os.path.join(INPUT_PATH, TEST_SET)
test_cls_sub_folders =  os.listdir(test_set_path)
test_cls_sub_folders.sort()
print(test_cls_sub_folders)

['ADVENT', 'LAM_PROP', 'MUSC_MUC', 'MUSC_PROP', 'REGR_TU', 'SH_MAG', 'SH_OES', 'SUBMUC', 'SUB_GL', 'TUMOR', 'ULCUS']


In [13]:
dict_of_alias_clsname_to_actual_clsname = {
  'ADVENT':'adventitia',
  'LAM_PROP':'lamina propria mucosae',
  'MUSC_MUC':'muscularis mucosae',
  'MUSC_PROP':'muscularis propria',
  'REGR_TU':'regression tissue',
  'SH_MAG':'mucosa gastric',
  'SH_OES':'mucosa oesophagus',
  'SUBMUC':'submucosa',
  'SUB_GL':'submucosal glands',
  'TUMOR':'tumour',
  'ULCUS':'ulceration'
}

In [8]:
dict_of_classname_to_filenames['test'] = {}
for cls_sub_folder in test_cls_sub_folders:
    cls_sub_path = os.path.join(test_set_path, cls_sub_folder)
    list_of_files = os.listdir(cls_sub_path)
    list_of_files.sort()
    print(f'{cls_sub_folder}: {len(list_of_files)} files.')
    dict_of_classname_to_filenames['test'][cls_sub_folder] = [os.path.join(TEST_SET, cls_sub_folder, file_name) for file_name in list_of_files]

ADVENT: 39258 files.
LAM_PROP: 1325 files.
MUSC_MUC: 867 files.
MUSC_PROP: 35581 files.
REGR_TU: 39464 files.
SH_MAG: 15345 files.
SH_OES: 9255 files.
SUBMUC: 9447 files.
SUB_GL: 918 files.
TUMOR: 26222 files.
ULCUS: 505 files.


In [10]:
dict_of_classname_to_filenames['train'] = {}
for train_set in TRAIN_SET:
    train_set_path = os.path.join(INPUT_PATH, train_set)
    train_cls_sub_folders = os.listdir(train_set_path)
    train_cls_sub_folders.sort()
    print(f'=== {train_set} ===')

    for cls_sub_folder in train_cls_sub_folders:
        cls_sub_path = os.path.join(train_set_path, cls_sub_folder)
        list_of_files = os.listdir(cls_sub_path)
        list_of_files.sort()
        print(f'{cls_sub_folder}: {len(list_of_files)} files.')
        dict_of_classname_to_filenames['train'][cls_sub_folder] = [os.path.join(train_set, cls_sub_folder, file_name) for file_name in list_of_files]

=== VALSET1_UKK ===
ADVENT: 7377 files.
LAM_PROP: 73 files.
MUSC_MUC: 104 files.
MUSC_PROP: 7321 files.
REGR_TU: 5839 files.
SH_MAG: 2558 files.
SH_OES: 1499 files.
SUBMUC: 479 files.
SUB_GL: 59 files.
TUMOR: 9378 files.
ULCUS: 17 files.
=== VALSET2_WNS ===
ADVENT: 21483 files.
LAM_PROP: 503 files.
MUSC_MUC: 1571 files.
MUSC_PROP: 30630 files.
REGR_TU: 11187 files.
SH_MAG: 23026 files.
SH_OES: 6522 files.
SUBMUC: 9224 files.
SUB_GL: 429 files.
TUMOR: 17042 files.
ULCUS: 26 files.
=== VALSET3_TCGA ===
ADVENT: 3013 files.
LAM_PROP: 272 files.
MUSC_MUC: 409 files.
MUSC_PROP: 9826 files.
REGR_TU: 0 files.
SH_MAG: 3488 files.
SH_OES: 1285 files.
SUBMUC: 2967 files.
SUB_GL: 110 files.
TUMOR: 11221 files.
ULCUS: 205 files.


## Generate balanced weights

In [12]:
SPLIT_PATH = '../../datasets/7_ESCA/splits/class2images_{}.p'

split_ordered_dict = {}
for split_name in ['train', 'val', 'test']:
  print(f'--- Read {split_name} pickle file ---')

  objects_list = []
  pickle_path = SPLIT_PATH.format(split_name)
  with open(pickle_path, "rb") as openfile:
    while True:
      try:
        objects_list.append(pickle.load(openfile))
      except EOFError:
        break

  print('-> len(objects_list): ', len(objects_list)) # 1
  print('-> type(objects_list[0]): ', type(objects_list[0])) # dictionary of mapping from a class name to list of corresponding files
  cnt_dict = {}
  for key, value in objects_list[0].items():
    print(key, ' ', len(value))
    cnt_dict[key] = len(value)

  split_ordered_dict[split_name] = collections.OrderedDict(sorted(cnt_dict.items()))
  

--- Read train pickle file ---
-> len(objects_list):  1
-> type(objects_list[0]):  <class 'dict'>
tumour   30113
regression tissue   13621
muscularis propria   38222
submucosa   10136
adventitia   25498
mucosa gastric   23257
mucosa oesophagus   7445
muscularis mucosae   1667
submucosal glands   478
ulceration   198
lamina propria mucosae   678
--- Read val pickle file ---
-> len(objects_list):  1
-> type(objects_list[0]):  <class 'dict'>
mucosa gastric   5814
muscularis propria   9555
tumour   7528
muscularis mucosae   417
submucosa   2534
adventitia   6375
regression tissue   3405
mucosa oesophagus   1861
lamina propria mucosae   170
submucosal glands   120
ulceration   50
--- Read test pickle file ---
-> len(objects_list):  1
-> type(objects_list[0]):  <class 'dict'>
adventitia   39258
lamina propria mucosae   1325
muscularis mucosae   867
muscularis propria   35581
regression tissue   39464
mucosa gastric   15345
mucosa oesophagus   9255
submucosa   9447
submucosal glands   918
tum

In [22]:
print('----- train -----')
for key, item in split_ordered_dict['train'].items():
  print(item, end=' ')
print()
print('----- val -----')
for key, item in split_ordered_dict['val'].items():
  print(item, end=' ')
print()
print('----- test -----')
for key, item in split_ordered_dict['test'].items():
  print(item, end=' ')

----- train -----
25498 678 23257 7445 1667 38222 13621 10136 478 30113 198 
----- val -----
6375 170 5814 1861 417 9555 3405 2534 120 7528 50 
----- test -----
39258 1325 15345 9255 867 35581 39464 9447 918 26222 505 

In [18]:
# ===== TRAIN =====
print('===== TRAIN =====')
recreated_labels_list_train = []
recreated_labels_list_val = []
recreated_labels_list_test = []

for cls_idx, (key, item) in enumerate(split_ordered_dict['train'].items()):
  recreated_labels_list_train.extend([cls_idx]*item)
for cls_idx, (key, item) in enumerate(split_ordered_dict['val'].items()):
  recreated_labels_list_val.extend([cls_idx]*item)
for cls_idx, (key, item) in enumerate(split_ordered_dict['test'].items()):
  recreated_labels_list_test.extend([cls_idx]*item)

print(recreated_labels_list_train)
print(len(recreated_labels_list_train))
print(recreated_labels_list_val)
print(len(recreated_labels_list_val))
print(recreated_labels_list_test)
print(len(recreated_labels_list_test))
print('=> Total: ', len(recreated_labels_list_train) + len(recreated_labels_list_val) + len(recreated_labels_list_test))

===== TRAIN =====
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 

In [20]:
# === compute class weights ===
class_weights_train = compute_class_weight(class_weight="balanced", classes=np.unique(recreated_labels_list_train), y=recreated_labels_list_train)
class_weights_val = compute_class_weight(class_weight="balanced", classes=np.unique(recreated_labels_list_val), y=recreated_labels_list_val)
class_weights_test = compute_class_weight(class_weight="balanced", classes=np.unique(recreated_labels_list_test), y=recreated_labels_list_test)

print(*class_weights_train, sep=", ")
print(*class_weights_val, sep=", ")
print(*class_weights_test, sep=", ")

0.5394825975655845, 20.288683293108072, 0.5914661079557669, 1.8476463764576592, 8.251786006435077, 0.3598903059161549, 1.009891144022265, 1.3571159503479946, 28.777672118676303, 0.4568036154726289, 69.47337006427915
0.5394509803921569, 20.229411764705883, 0.5915032679738562, 1.847931219774315, 8.247002398081534, 0.35991627420198846, 1.0099853157121879, 1.3571428571428572, 28.658333333333335, 0.4568278427205101, 68.78
0.4126246416480254, 12.225523156089194, 1.05564145857401, 1.7502774912823535, 18.68375799517668, 0.4552659616598235, 0.410470762766526, 1.7147050049558783, 17.64577143988909, 0.6177567760589651, 32.07686768676868
